In [1]:
%pylab inline

Populating the interactive namespace from numpy and matplotlib


In [2]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

In [3]:
import pandas as pd
from glob import glob
from tqdm import tqdm
from skimage import io
import deepdish as dd

In [ ]:
from keras import backend as K
from keras.models import Model
from keras.optimizers import Adam

from keras.regularizers import l2
from keras.models import load_model
from keras.callbacks import ModelCheckpoint,EarlyStopping
from keras.layers import Input, BatchNormalization, Conv2D, Dense, Activation, Dropout

In [5]:
from keras.applications import Xception,VGG16,VGG19,ResNet50
from keras.applications import InceptionV3, InceptionResNetV2, MobileNet, DenseNet121, DenseNet169, DenseNet201

variants = ['Xception', 'VGG16', 'VGG19', 
            'ResNet50', 'InceptionV3', 'InceptionResNetV2', 
            'DenseNet121', 'DenseNet169', 'DenseNet201']

In [6]:
angle_range = 30
rotation_step = 5
dropout_rate = 0.25

In [8]:
def get_features(features_file):
    
    features = dd.io.load(features_file)
    
    return(features)


def get_model(model_path, reset_weights=False):
    
    model = load_model(model_path)
    
    if reset_weights:
        reset_weights(model)
        
    return(model)

def reset_weights(model_in):
    
    session = K.get_session()
    
    for layer in model_in.layers: 
        if hasattr(layer, 'kernel_initializer'):
            layer.kernel.initializer.run(session=session)

In [10]:
retrain = False

In [12]:
final_results = []

for variant in variants[1:2]:
    
    print(variant)
    run_version = 'model_new'+variant+'_'+str(angle_range)
    features_file = './01.features/model_'+variant+'_90_features.h5'

    features = get_features(features_file)
    
    data_x = features['data_x']
    data_y = features['data_y']
    data_paths = features['data_paths']
    data_rotation = features['data_rotation']

    subset = (np.absolute(data_rotation)<=60)

    train_x = data_x[subset]
    train_y = data_y[subset]
    data_paths = data_paths[subset]
    data_rotation = data_rotation[subset]
    
    model = get_model('./02.model/'+run_version+'.h5')
    data_pred = model.predict(train_x,batch_size=10,verbose=1)
    
    model_results = pd.DataFrame({
                            'model':run_version,
                            'image_paths':data_paths,
                            'image_rotations':data_rotation,
                            'y_actual':train_y.flatten(),
                            'y_predicted':data_pred.flatten()
                            })
    
    final_results += [model_results]

VGG16
10164/10164 [==============================] - 2s 201us/step


In [24]:
final_predictions = pd.concat(final_results)
#final_predictions.to_csv('./03.results/combined_output_new_'+str(angle_range)+'.txt',sep='|', index=False)

In [26]:
final_predictions['delta'] = final_predictions.y_actual-final_predictions.y_predicted

In [35]:
final_predictions[final_predictions.image_rotations.between(-31,31)].groupby('model')['delta'].apply(lambda x: np.sqrt((x**2).mean()))

model
model_newDenseNet121_30          2.604867
model_newDenseNet169_30          2.798947
model_newDenseNet201_30          2.798945
model_newInceptionResNetV2_30    3.083203
model_newInceptionV3_30          3.696053
model_newResNet50_30             2.792018
model_newVGG16_30                2.055010
model_newVGG19_30                1.904698
model_newXception_30             4.033000
Name: delta, dtype: float64